In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/bengaluru_house_prices.csv')
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00
...,...,...,...,...,...,...,...,...,...
13315,Built-up Area,Ready To Move,Whitefield,5 Bedroom,ArsiaEx,3453,4.0,0.0,231.00
13316,Super built-up Area,Ready To Move,Richards Town,4 BHK,NaN,3600,5.0,NaN,400.00
13317,Built-up Area,Ready To Move,Raja Rajeshwari Nagar,2 BHK,Mahla T,1141,2.0,1.0,60.00
13318,Super built-up Area,18-Jun,Padmanabhanagar,4 BHK,SollyCl,4689,4.0,1.0,488.00


In [ ]:
df.info()

display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


,bath,balcony,price
count,13247.000000,12711.000000,13320.000000
mean,2.692610,1.584376,112.565627
std,1.341458,0.817263,148.971674
min,1.000000,0.000000,8.000000
25%,2.000000,1.000000,50.000000
50%,2.000000,2.000000,72.000000
75%,3.000000,2.000000,120.000000
max,40.000000,3.000000,3600.000000


In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique 'size' values:", df['size'].unique())
print("\nSample 'total_sqft' values:", df['total_sqft'].sample(20).values)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB

Missing values:
 area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

Duplicate rows: 529

Unique 'size' values: ['2 BHK' '4 Bedroom' '3 BHK' '4 BHK' '6 Bedroom' '3 Bedroom' '1 BHK'
 '1 RK' '1 Bedroom' '8 Bedroom' 

In [ ]:
# Drop duplicates
df = df.drop_duplicates()

# Extract the number of rooms from 'size' (e.g. "2 BHK" -> 2, "4 Bedroom" -> 4)
df['bhk'] = df['size'].str.extract('(\d+)').astype(float)

# Check for any total_sqft values that aren't plain numbers (like "1000-1200")
def is_plain_number(x):
    try:
        float(x)
        return True
    except:
        return False

non_numeric = df[~df['total_sqft'].apply(is_plain_number)]
print("Rows with non-numeric total_sqft:", len(non_numeric))
non_numeric['total_sqft'].head(10)

Rows with non-numeric total_sqft: 246


<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_471/351057038.py:5: SyntaxWarning: invalid escape sequence '\d'
  df['bhk'] = df['size'].str.extract('(\d+)').astype(float)


,total_sqft
30,2100 - 2850
56,3010 - 3410
81,2957 - 3450
122,3067 - 8156
137,1042 - 1105
165,1145 - 1340
188,1015 - 1540
224,1520 - 1740
410,34.46Sq. Meter
549,1195 - 1440


In [ ]:
df = df.drop_duplicates().copy()

# Fix the earlier warning
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)

# Convert total_sqft: ranges become the average, anything else becomes NaN
def convert_sqft(x):
    try:
        return float(x)
    except:
        if '-' in str(x):
            parts = x.split('-')
            try:
                return (float(parts[0]) + float(parts[1])) / 2
            except:
                return None
        return None  # weird units like "Sq. Meter" - can't safely convert, drop later

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

print("Rows now missing total_sqft:", df['total_sqft'].isnull().sum())

Rows now missing total_sqft: 46


In [ ]:
# Drop rows where total_sqft couldn't be converted
df = df.dropna(subset=['total_sqft'])

# Drop 'society' - over 40% missing, not worth trying to fill
df = df.drop('society', axis=1)

# Drop the few rows missing location, size/bhk, or bath (small numbers, safe to drop)
df = df.dropna(subset=['location', 'bhk', 'bath'])

# Fill missing balcony with the median (609 missing is more, so we fill rather than drop)
df['balcony'] = df['balcony'].fillna(df['balcony'].median())

# Look at the bhk outliers before deciding a cutoff
print(df['bhk'].value_counts().sort_index())

bhk
1.0      622
2.0     5234
3.0     4616
4.0     1371
5.0      343
6.0      220
7.0       99
8.0       88
9.0       52
10.0      14
11.0       4
12.0       1
13.0       1
14.0       1
16.0       1
18.0       1
19.0       1
27.0       1
43.0       1
Name: count, dtype: int64


In [ ]:
# Drop the extreme bhk outliers (10 and above)
df = df[df['bhk'] <= 10]

print("Rows remaining:", len(df))
df.describe()

Rows remaining: 12659


,total_sqft,bath,balcony,price,bhk
count,12659.000000,12659.000000,12659.000000,12659.000000,12659.000000
mean,1563.014765,2.694763,1.601232,113.851680,2.801327
std,1254.429805,1.264091,0.808541,151.585979,1.201967
min,1.000000,1.000000,0.000000,8.000000,1.000000
25%,1100.000000,2.000000,1.000000,50.000000,2.000000
50%,1280.000000,2.000000,2.000000,72.600000,3.000000
75%,1685.000000,3.000000,2.000000,120.000000,3.000000
max,52272.000000,14.000000,3.000000,3600.000000,10.000000


In [ ]:
df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']  # price is in lakhs, so *100000 for rupees
print(df['price_per_sqft'].describe())

count    1.265900e+04
mean     8.033790e+03
std      1.089811e+05
min      2.678298e+02
25%      4.298610e+03
50%      5.483631e+03
75%      7.400532e+03
max      1.200000e+07
Name: price_per_sqft, dtype: float64


In [ ]:
# Sanity check: a real bedroom needs at least ~300 sqft. Anything below is bad data.
df = df[(df['total_sqft'] / df['bhk']) >= 300]

# Remove extreme price_per_sqft outliers using the IQR method (a standard, defensible cutoff)
Q1 = df['price_per_sqft'].quantile(0.25)
Q3 = df['price_per_sqft'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[(df['price_per_sqft'] >= lower) & (df['price_per_sqft'] <= upper)]

print("Rows remaining:", len(df))
df['price_per_sqft'].describe()

Rows remaining: 10647


,price_per_sqft
count,10647.000000
mean,5311.295947
std,1588.885061
min,729.860414
25%,4126.825397
50%,5081.300813
75%,6307.255245
max,9912.886753


In [ ]:
# Drop price_per_sqft (data leakage) and size (redundant with bhk)
df = df.drop(['price_per_sqft', 'size'], axis=1)

# Simplify availability: 1 if "Ready To Move", 0 otherwise (specific dates = not ready yet)
df['ready_to_move'] = (df['availability'] == 'Ready To Move').astype(int)
df = df.drop('availability', axis=1)

# Check how many unique locations we're dealing with
print("Unique locations:", df['location'].nunique())
print(df['location'].value_counts().head(10))

Unique locations: 1078
location
Whitefield               470
Sarjapur  Road           348
Electronic City          274
Kanakpura Road           238
Thanisandra              225
Yelahanka                197
Marathahalli             157
Raja Rajeshwari Nagar    153
Uttarahalli              149
Hennur Road              142
Name: count, dtype: int64


In [ ]:
# Group locations with 10 or fewer listings into 'other'
location_counts = df['location'].value_counts()
rare_locations = location_counts[location_counts <= 10].index
df['location'] = df['location'].apply(lambda x: 'other' if x in rare_locations else x)

print("Locations after grouping:", df['location'].nunique())

# Now one-hot encode location and area_type
df = pd.get_dummies(df, columns=['location', 'area_type'], drop_first=True)

print("Total columns now:", df.shape[1])
df.head()

Locations after grouping: 198
Total columns now: 206


,total_sqft,bath,balcony,price,bhk,ready_to_move,location_1st Phase JP Nagar,location_5th Phase JP Nagar,location_6th Phase JP Nagar,location_7th Phase JP Nagar,...,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur,location_other,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,1056.0,2.0,1.0,39.07,2.0,0,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1,2600.0,5.0,3.0,120.00,4.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,1440.0,2.0,3.0,62.00,3.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1521.0,3.0,1.0,95.00,3.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4,1200.0,2.0,1.0,51.00,2.0,1,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)  # no activation - we want a raw number (price), not 0-1
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        13,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,809 (61.75 KB)

 Trainable params: 15,809 (61.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 941.0443 - mae: 18.5260 - val_loss: 727.2236 - val_mae: 17.0188
Epoch 2/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 930.5471 - mae: 18.2089 - val_loss: 723.2028 - val_mae: 16.9246
Epoch 3/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 934.5900 - mae: 18.2541 - val_loss: 717.9954 - val_mae: 16.8906
Epoch 4/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 921.7476 - mae: 18.0607 - val_loss: 725.8015 - val_mae: 17.0483
Epoch 5/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 904.1480 - mae: 18.0509 - val_loss: 716.6280 - val_mae: 16.9899
Epoch 6/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 941.3830 - mae: 18.0007 - val_loss: 733.4878 - val_mae: 17.4340
Epoch 7/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 937.0764 - mae: 18.3927 - val_loss: 739.4367 - val_mae: 16.9769
Epoch 8/200
213/213 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 888.7770 - mae: 17.9074 - val_loss: 721.1970 - val_mae: 16.84

In [ ]:
from sklearn.metrics import r2_score

test_loss, test_mae = model.evaluate(X_test_scaled, y_test)
predictions = model.predict(X_test_scaled)
r2 = r2_score(y_test, predictions)

print(f"Test MAE: {test_mae:.2f} lakhs")
print(f"R² score: {r2:.4f}")

67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 959.8056 - mae: 18.6268
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
Test MAE: 18.63 lakhs
R² score: 0.8660
